In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
df = pd.read_csv("/Users/brandonsmith/data-515/DATA-515-Final-Project/data/startup_founder_burnout_2026_project.csv")

# Burnout Percentage

# Creating Feature and Target

In [8]:
# Creating feature and target

feature = df.drop(['Startup_Failure_Flag', 'Shutdown_Probability', 'Shutdown_Risk'], axis=1)
target = df['Startup_Failure_Flag']

In [9]:
#Train Test Split and Scaling
featues_binary = pd.get_dummies(feature, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(featues_binary, target, test_size=0.2, random_state=42)

scaler = StandardScaler()

x_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to retain feature names
x_scaled = pd.DataFrame(x_scaled, columns=X_train.columns)
x_test_scaled = pd.DataFrame(x_test_scaled, columns=X_test.columns)

: 

# Logisitic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

lrmodel = LogisticRegression(random_state=42)
lrmodel.fit(x_scaled, y_train)
y_pred = lrmodel.predict(x_test_scaled)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_auc_score
import matplotlib.pyplot as plt

y_pred_proba_lr = lrmodel.predict_proba(x_test_scaled)[:, 1]

# Calculate Precision-Recall curve
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_pred_proba_lr)

fig_lr, ax_lr = plt.subplots(figsize=(8, 6))
ax_lr.plot(recall_lr, precision_lr, label='Logistic Regression')
ax_lr.set_xlabel('Recall')
ax_lr.set_ylabel('Precision')
ax_lr.set_title('Precision-Recall Curve for Logistic Regression')
ax_lr.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
print(f"Logistic Regression ROC AUC Score: {roc_auc_lr:.4f}")

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=lrmodel.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=lrmodel.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix (Logistic Regression)')
y_pred_proba = lrmodel.predict_proba(x_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")
plt.show()

## Accuracy : 73.18%

## ROC Score: 0.725

# Logistic Regression w/ Weighted Balance Classes

In [ ]:
#With Class Balance
lrmodel_weighted = LogisticRegression(class_weight = 'balanced', random_state=42)
lrmodel_weighted.fit(x_scaled, y_train)

y_pred = lrmodel_weighted.predict(x_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=lrmodel_weighted.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=lrmodel_weighted.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix (Logistic Regression)')
y_pred_proba = lrmodel_weighted.predict_proba(x_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")
plt.show()

## Accuracy: 67.34%

## ROC Score: 0.725

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

new_threshold_lr = 0.5

y_pred_lr_tuned_threshold = (y_pred_proba_lr >= new_threshold_lr).astype(int)

accuracy_lr_tuned = accuracy_score(y_test, y_pred_lr_tuned_threshold)

print(f'Logistic Regression Accuracy (Threshold {new_threshold_lr}): {accuracy_lr_tuned * 100:.2f}%')

print("\nClassification Report (Logistic Regression, tuned threshold):")
print(classification_report(y_test, y_pred_lr_tuned_threshold))

cm_lr_tuned = confusion_matrix(y_test, y_pred_lr_tuned_threshold, labels=lrmodel.classes_)
disp_lr_tuned = ConfusionMatrixDisplay(confusion_matrix=cm_lr_tuned, display_labels=lrmodel.classes_)
disp_lr_tuned.plot(cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix (Logistic Regression, Threshold {new_threshold_lr})')
plt.show()

# XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

In [ ]:
#XG Boost Model

params = {
    'eval_metric': 'logloss',
    'random_state': 42
}

xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(x_scaled, y_train)
predxgb = xgb_model.predict(x_test_scaled)
xgb_accuracy_score = accuracy_score(y_test, predxgb)

print('Accuarcy of model is:', xgb_accuracy_score * 100)
y_pred_proba_xgb = xgb_model.predict_proba(x_test_scaled)[:, 1]
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print("ROC Score:", roc_auc_xgb)

## Accuracy: 73.06%

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay


# Predict positive class
pred_proba_xgb = xgb_model.predict_proba(x_test_scaled)[:, 1]

# ROC AUC score
roc_auc = roc_auc_score(y_test, pred_proba_xgb)
print(f'ROC AUC Score: {roc_auc:.4f}')
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(xgb_model, x_test_scaled, y_test, ax=ax, name='XGBoost')
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## ROC Score: 0.695

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

#Classification Report
print("\nClassification Report:")
print(classification_report(y_test, predxgb))

#Confusion Matrix
cm = confusion_matrix(y_test, predxgb, labels=xgb_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=xgb_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

In [ ]:
burnout_target = df['Founder_Burnout_Flag']
burnout_features = df.drop(['Founder_Burnout_Flag','Startup_Failure_Flag', 'Shutdown_Probability', 'Shutdown_Risk'], axis = 1)
burnout_features_binary = pd.get_dummies(burnout_features, drop_first = True)

X_train_b, X_test_b, y_train_b, _y_test_b = train_test_split(burnout_features_binary, burnout_target, test_size = 0.2, random_state = 42)

scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

burnout_prob = xgb_model.predict_proba(scaler_b.transform(burnout_features_binary))[:,1]
df['Burnout_Percentage'] = burnout_prob * 100

# XGBoost Grid and Random Search

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
#Grid Search

param_grid = {
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100,1000],
}
grid_search = GridSearchCV(xgb_model, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(x_scaled, y_train)
print("Grid Search Best Params:")
print(grid_search.best_params_)
test_predictions = grid_search.predict(x_test_scaled)
final_test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")
print("ROC Score:", grid_search.best_score_)

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
# Random Search

param_dist = {
    'max_depth': randint(1, 10),
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': randint(50, 1000),
    'subsample': [0.6, 0.8,0.9, 1.0],
    'colsample_bytree': [0.6, 0.8,0.9, 1.0],
    'gamma': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_lambda': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}
random_search = RandomizedSearchCV(xgb_model, param_dist, n_iter=10, cv=5, scoring='roc_auc', n_jobs=-1)
random_search.fit(x_scaled, y_train)
print("Random Search Best Params:")
print(random_search.best_params_)
test_predictions = random_search.predict(x_test_scaled)
final_test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")
print("ROC Score:", random_search.best_score_)

In [ ]:
#XG Boost Model with new parameters

new_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42,
    'colsample_bytree': 0.8,
    'learning_rate': 0.01,
    'max_depth': 2,
    'n_estimators': 669,
    'subsample': 0.8,
    'gamma': 0.4,
    'reg_alpha': 0.1,
    'reg_lambda': 0.2
}

new_xgb_model = xgb.XGBClassifier(**new_params)
new_xgb_model.fit(x_scaled, y_train)
newpredxgb = new_xgb_model.predict(x_test_scaled)
newxgb_accuracy_score = accuracy_score(y_test, newpredxgb)

print('Accuarcy of model is:', newxgb_accuracy_score * 100)
print("ROC Score:", random_search.best_score_)

## Accuracy: 73.16%

## ROC Score: 0.727

In [ ]:
#XGB Precion Curve
from sklearn.metrics import precision_recall_curve, roc_auc_score
import matplotlib.pyplot as plt

y_pred_new_xgb_model = new_xgb_model.predict_proba(x_test_scaled)[:, 1]

# Calculate Precision-Recall curve
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_pred_new_xgb_model)

fig_lr, ax_lr = plt.subplots(figsize=(8, 6))
ax_lr.plot(recall_xgb, precision_xgb, label='XGBoost')
ax_lr.set_xlabel('Recall')
ax_lr.set_ylabel('Precision')
ax_lr.set_title('Precision-Recall Curve for XGB Boost')
ax_lr.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

roc_auc_xgb = roc_auc_score(y_test, y_pred_new_xgb_model)
print(f"XGB Boost ROC AUC Score: {roc_auc_xgb:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

new_threshold_xgb = 0.55

y_pred_xgb_tuned_threshold = (y_pred_new_xgb_model >= new_threshold_xgb).astype(int)

accuracy_xgb_tuned = accuracy_score(y_test, y_pred_xgb_tuned_threshold)

print(f'XGBoost Accuracy (Threshold {new_threshold_xgb}): {accuracy_xgb_tuned * 100:.2f}%')
print("ROC Score:", random_search.best_score_)

print("\nClassification Report (XGBoost, tuned threshold):")
print(classification_report(y_test, y_pred_xgb_tuned_threshold))

cm_xgb_tuned = confusion_matrix(y_test, y_pred_xgb_tuned_threshold, labels=new_xgb_model.classes_)
disp_xgb_tuned = ConfusionMatrixDisplay(confusion_matrix=cm_xgb_tuned, display_labels=new_xgb_model.classes_)
disp_xgb_tuned.plot(cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix (XGBoost, Threshold {new_threshold_xgb})')
plt.show()